# K-Means: Zero to Hero
### Part 1 — K-Means clustering and its variations

This notebook is a hands-on tour of **K-Means** and the family of algorithms around it.

**What you'll see:**
1. K-Means intuition + a from-scratch NumPy implementation (Lloyd's algorithm)
2. scikit-learn `KMeans` on synthetic blobs
3. Choosing **k**: the Elbow method and Silhouette analysis
4. **K-Means++** vs random initialization
5. **MiniBatchKMeans** — scaling to large data
6. **Bisecting K-Means** — a divisive hierarchical variant
7. Where K-Means fails, and better alternatives (**GaussianMixture**, **Spectral/DBSCAN**)
8. A real application: **image color quantization**
9. Evaluation metrics round-up

> Runtime: **CPU** is enough. `Runtime → Run all`.

In [ ]:
# If running in a fresh Colab, these are already installed. Uncomment if needed.
# !pip install -q scikit-learn matplotlib numpy pandas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MiniBatchKMeans, BisectingKMeans, SpectralClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             davies_bouldin_score, adjusted_rand_score)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (7, 5)
print("Libraries loaded.")

## 1. The intuition

K-Means partitions data into **k** groups by minimizing **within-cluster sum of squares (inertia)**.

**Lloyd's algorithm** repeats two steps until convergence:
1. **Assign** each point to its nearest centroid.
2. **Update** each centroid to the mean of the points assigned to it.

Let's generate some well-separated blobs to work with.

In [ ]:
X, y_true = make_blobs(n_samples=500, centers=4, cluster_std=0.90,
                       random_state=RANDOM_STATE)

plt.scatter(X[:, 0], X[:, 1], s=18, c='gray')
plt.title("Raw data (4 true blobs, labels hidden)")
plt.xlabel("feature 0"); plt.ylabel("feature 1")
plt.show()

## 2. K-Means from scratch (NumPy)

Implementing Lloyd's algorithm ourselves makes the mechanics concrete before we lean on scikit-learn.

In [ ]:
def kmeans_scratch(X, k, n_iters=100, tol=1e-4, seed=0):
    rng = np.random.default_rng(seed)
    # init: pick k random distinct points as centroids
    centroids = X[rng.choice(len(X), size=k, replace=False)].copy()
    for it in range(n_iters):
        # ASSIGN: distance from each point to each centroid -> nearest
        dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
        labels = dists.argmin(axis=1)
        # UPDATE: move centroids to the mean of their members
        new_centroids = np.array([
            X[labels == j].mean(axis=0) if np.any(labels == j) else centroids[j]
            for j in range(k)
        ])
        shift = np.linalg.norm(new_centroids - centroids)
        centroids = new_centroids
        if shift < tol:
            print(f"Converged at iteration {it} (shift={shift:.2e})")
            break
    inertia = sum(((X[labels == j] - centroids[j])**2).sum() for j in range(k))
    return labels, centroids, inertia

labels_s, cent_s, inertia_s = kmeans_scratch(X, k=4, seed=RANDOM_STATE)
print("Scratch inertia:", round(inertia_s, 2))

plt.scatter(X[:, 0], X[:, 1], c=labels_s, s=18, cmap='viridis')
plt.scatter(cent_s[:, 0], cent_s[:, 1], c='red', s=200, marker='X', edgecolor='k')
plt.title("From-scratch K-Means (k=4)")
plt.show()

## 3. scikit-learn KMeans

Same result, far less code — plus speed and robustness. Note `inertia_` matches our scratch value closely.

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE)
labels_sk = km.fit_predict(X)
print("sklearn inertia:", round(km.inertia_, 2))
print("Adjusted Rand Index vs true labels:", round(adjusted_rand_score(y_true, labels_sk), 3))

plt.scatter(X[:, 0], X[:, 1], c=labels_sk, s=18, cmap='viridis')
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            c='red', s=200, marker='X', edgecolor='k')
plt.title("sklearn KMeans (k=4)")
plt.show()

## 4. Choosing k — Elbow method & Silhouette

In practice we don't know **k**. Two standard tools:
- **Elbow**: plot inertia vs k; look for the "elbow" where gains flatten.
- **Silhouette**: average silhouette score (−1..1); higher is better-separated.

In [ ]:
ks = range(2, 10)
inertias, sils = [], []
for k in ks:
    m = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    inertias.append(m.inertia_)
    sils.append(silhouette_score(X, m.labels_))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(list(ks), inertias, 'o-'); ax[0].set_title("Elbow (inertia)")
ax[0].set_xlabel("k"); ax[0].set_ylabel("inertia")
ax[1].plot(list(ks), sils, 'o-', color='green'); ax[1].set_title("Silhouette score")
ax[1].set_xlabel("k"); ax[1].set_ylabel("avg silhouette")
plt.show()
print("Best k by silhouette:", list(ks)[int(np.argmax(sils))])

In [ ]:
# Silhouette diagram for k=4 (per-sample view)
k = 4
m = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
sample_sil = silhouette_samples(X, m.labels_)
y_lower = 10
plt.figure(figsize=(7,5))
for i in range(k):
    vals = np.sort(sample_sil[m.labels_ == i])
    y_upper = y_lower + len(vals)
    plt.fill_betweenx(np.arange(y_lower, y_upper), 0, vals, alpha=0.7)
    y_lower = y_upper + 10
plt.axvline(silhouette_score(X, m.labels_), color='red', ls='--', label='avg')
plt.title(f"Silhouette plot (k={k})"); plt.xlabel("silhouette coefficient")
plt.legend(); plt.show()

## 5. K-Means++ vs random initialization

Random init can land in bad local minima. **K-Means++** spreads initial centroids out, giving
faster convergence and lower inertia on average. scikit-learn uses `init='k-means++'` by default.

In [ ]:
results = {}
for init in ['random', 'k-means++']:
    # single init to expose the difference (n_init=1)
    m = KMeans(n_clusters=4, init=init, n_init=1, random_state=1).fit(X)
    results[init] = m.inertia_
print("Inertia with n_init=1:")
for k_, v in results.items():
    print(f"  {k_:>10}: {v:.2f}")
print("\nk-means++ typically matches or beats random, especially on harder data.")

## 6. MiniBatchKMeans — scaling up

For large datasets, `MiniBatchKMeans` updates centroids on small random batches. It's much
faster with a small, usually acceptable, loss in quality.

In [ ]:
import time
X_big, _ = make_blobs(n_samples=100_000, centers=6, cluster_std=1.0,
                      random_state=RANDOM_STATE)

t0 = time.time()
km_full = KMeans(n_clusters=6, n_init=5, random_state=RANDOM_STATE).fit(X_big)
t_full = time.time() - t0

t0 = time.time()
km_mb = MiniBatchKMeans(n_clusters=6, batch_size=1024, n_init=5,
                        random_state=RANDOM_STATE).fit(X_big)
t_mb = time.time() - t0

print(f"KMeans      : time={t_full:.2f}s  inertia={km_full.inertia_:.0f}")
print(f"MiniBatch   : time={t_mb:.2f}s  inertia={km_mb.inertia_:.0f}")
print(f"Speedup     : {t_full/t_mb:.1f}x")

## 7. Bisecting K-Means

A **divisive hierarchical** variant: start with one cluster and repeatedly split the worst
cluster with 2-means. Often produces more balanced clusters and better results than plain
K-Means on some structures.

In [ ]:
bk = BisectingKMeans(n_clusters=4, random_state=RANDOM_STATE).fit(X)
print("Bisecting KMeans inertia:", round(bk.inertia_, 2))
plt.scatter(X[:, 0], X[:, 1], c=bk.labels_, s=18, cmap='viridis')
plt.scatter(bk.cluster_centers_[:, 0], bk.cluster_centers_[:, 1],
            c='red', s=200, marker='X', edgecolor='k')
plt.title("Bisecting K-Means (k=4)")
plt.show()

## 8. Where K-Means fails — and what to use instead

K-Means assumes **round, equally-sized, linearly-separable** clusters. On non-convex shapes
(like two moons) it fails. **Spectral clustering** and **DBSCAN** handle these; **GaussianMixture**
handles elliptical clusters.

In [ ]:
Xm, ym = make_moons(n_samples=400, noise=0.06, random_state=RANDOM_STATE)

methods = {
    "KMeans (fails)": KMeans(n_clusters=2, n_init=10, random_state=RANDOM_STATE).fit_predict(Xm),
    "Spectral": SpectralClustering(n_clusters=2, affinity='nearest_neighbors',
                                    random_state=RANDOM_STATE).fit_predict(Xm),
    "DBSCAN": DBSCAN(eps=0.2, min_samples=5).fit_predict(Xm),
}

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, (name, lab) in zip(ax, methods.items()):
    a.scatter(Xm[:, 0], Xm[:, 1], c=lab, s=15, cmap='viridis')
    a.set_title(name)
plt.show()

In [ ]:
# GaussianMixture on elliptical / overlapping blobs (soft assignment)
Xe, _ = make_blobs(n_samples=500, centers=3, cluster_std=[1.0, 2.5, 0.5],
                   random_state=RANDOM_STATE)
gmm = GaussianMixture(n_components=3, random_state=RANDOM_STATE).fit(Xe)
plt.scatter(Xe[:, 0], Xe[:, 1], c=gmm.predict(Xe), s=15, cmap='viridis')
plt.title("GaussianMixture (handles unequal variances)")
plt.show()

## 9. Real application — image color quantization

K-Means on pixel RGB values compresses an image to **k colors**. Classic, visual, and intuitive.

In [ ]:
from sklearn.datasets import load_sample_image
china = load_sample_image("china.jpg")
china = np.array(china, dtype=np.float64) / 255
w, h, d = china.shape
pixels = china.reshape(-1, d)

fig, ax = plt.subplots(1, 4, figsize=(18, 5))
ax[0].imshow(china); ax[0].set_title("Original (96K+ colors)"); ax[0].axis('off')
for i, k in enumerate([16, 8, 4], start=1):
    km_img = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE).fit(pixels)
    quantized = km_img.cluster_centers_[km_img.labels_].reshape(w, h, d)
    ax[i].imshow(quantized); ax[i].set_title(f"{k} colors"); ax[i].axis('off')
plt.show()

## 10. Evaluation metrics round-up

On the Iris dataset (which has ground-truth labels) we compare three metrics.
Standardizing features first is important for distance-based methods.

In [ ]:
iris = load_iris()
Xi = StandardScaler().fit_transform(iris.data)
km_iris = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit(Xi)

print("Inertia (lower better)          :", round(km_iris.inertia_, 2))
print("Silhouette (higher better)      :", round(silhouette_score(Xi, km_iris.labels_), 3))
print("Davies-Bouldin (lower better)   :", round(davies_bouldin_score(Xi, km_iris.labels_), 3))
print("Adjusted Rand vs truth (1=perfect):", round(adjusted_rand_score(iris.target, km_iris.labels_), 3))

## Summary

| Variant | When to use |
|---------|-------------|
| **KMeans** | Default; round, separated clusters |
| **K-Means++ init** | Always — better/faster convergence |
| **MiniBatchKMeans** | Large datasets, need speed |
| **BisectingKMeans** | Want hierarchy / more balanced clusters |
| **GaussianMixture** | Elliptical clusters, soft assignment |
| **Spectral / DBSCAN** | Non-convex shapes, arbitrary geometry |

**Key takeaways:** standardize features, use k-means++, pick *k* with elbow **and** silhouette,
and remember K-Means' round-cluster assumption — switch methods when it breaks.